In [1]:
import pandas as pd
import numpy as np

In [2]:
EPIC_CSV_PATH = "../data/silver/epic_games_clean.csv"
STEAM_CSV_PATH = "../data/silver/steam_clean.csv"
IGDB_CSV_PATH = "../data/silver/igdb_clean_post.csv"

epic_df = pd.read_csv(EPIC_CSV_PATH)
steam_df = pd.read_csv(STEAM_CSV_PATH)
igdb_df = pd.read_csv(IGDB_CSV_PATH)

In [3]:
# Normalize title / name to a join key (still dtype object)
def make_join_key(s: pd.Series) -> pd.Series:
    return (
        s.astype(str)          # ensure object/string
         .str.strip()
         .str.lower()
    )

def normalize_str_or_list(col: pd.Series) -> pd.Series:
    def f(v):
        if isinstance(v, (list, tuple, set)):
            return ", ".join(map(str, v))
        elif pd.isna(v):
            return np.nan
        else:
            return str(v)
    return col.apply(f)

def to_list(x):
    if isinstance(x, str):
        return x.split("|")      # split pipe string
    elif isinstance(x, list):
        return x                 # already a list
    else:
        return []                # None, NaN, float → empty list
    
def coalesce(series):
    """Return first non-null value in the group."""
    for v in series:
        if not is_null(v):
            return v
    return None

def is_null(v):
    """Robust null check that works for arrays/lists/objects."""
    if v is None:
        return True
    if isinstance(v, float) and np.isnan(v):
        return True
    return False

def merge_platforms(series: pd.Series):
    out = []
    for v in series.dropna():
        out.extend(v)
    # keep unique order
    return list(dict.fromkeys(out))


In [5]:
steam = steam_df.copy()

steam["steam_ID"] = make_join_key(steam["steam_appid"])
steam["epic_ID"] = pd.NaT
steam["igdb_ID"] = pd.NaT
steam["title"] = make_join_key(steam["name"])
steam["platform"] = [["steam"]] * len(steam)
steam["developers"] = normalize_str_or_list(steam["developers"])
steam["publishers_steam"] = normalize_str_or_list(steam["publishers"])
steam["publishers_epic"] = pd.NaT
steam["categories"] = steam["categories"].apply(to_list)
steam["genres"] = steam["genres"].apply(to_list)
steam["categories_steam"] = steam.apply(
    lambda row: (row["categories"] or []) + (row["genres"] or []),
    axis=1
)
steam.drop(columns=["categories", "genres"], inplace=True)
steam["effective_date_steam"] = pd.to_datetime(steam["effective_date"])
steam["effective_date_epic"] = pd.NaT
steam["effective_date_igdb"] = pd.NaT

# derive original price_steam from final_price & discount_percent
steam["price_steam"] = steam["final_price_usd"]
steam["price_epic"] = np.nan
steam["discount_percent_steam"] = steam["discount_percent"]
steam["discount_percent_epic"] = np.nan
steam["categories_epic"] = [[] for _ in range(len(steam))]
steam["categories_igdb"] = [[] for _ in range(len(steam))]


In [7]:
epic = epic_df.copy()

epic["steam_ID"] = pd.NaT
epic["epic_ID"] = make_join_key(epic["id"])
epic["igdb_ID"] = pd.NaT
epic["title"] = make_join_key(epic["title"])
epic["platform"] = [["epic"]] * len(epic)

epic["developers"] = np.nan     # not available
epic["publishers_steam"] = pd.NaT
epic["publishers_epic"] = epic["seller_name"].astype(str)

epic["categories_epic"] = epic["tags_joined"].apply(to_list)
epic["categories_steam"] = [[] for _ in range(len(epic))]
epic["categories_igdb"] = [[] for _ in range(len(epic))]

epic["effective_date_epic"] = pd.to_datetime(epic["effective_date"])
epic["effective_date_steam"] = pd.NaT
epic["effective_date_igdb"] = pd.NaT

epic["price_epic"] = epic["original_price"]
epic["price_steam"] = np.nan
epic["discount_percent_steam"] = np.nan
epic["discount_percent_epic"] = (
    1 - (epic["discount_price"] / epic["original_price"])
)


In [8]:
igdb = igdb_df.copy()

igdb["steam_ID"] = pd.NaT
igdb["epic_ID"] = pd.NaT
igdb["igdb_ID"] = make_join_key(igdb["id"])
igdb["title"] = make_join_key(igdb["name"])
igdb["platform"] = [["igdb"]] * len(igdb)

igdb["developers"] = np.nan
igdb["publishers_steam"] = np.nan
igdb["publishers_epic"] = np.nan

igdb["categories_igdb"] = igdb["genres"].apply(to_list)
igdb["categories_steam"] = [[] for _ in range(len(igdb))]
igdb["categories_epic"]  = [[] for _ in range(len(igdb))]

igdb["effective_date_igdb"] = pd.to_datetime(igdb["release_date"])
igdb["effective_date_steam"] = pd.NaT
igdb["effective_date_epic"]  = pd.NaT

igdb["price_steam"] = np.nan
igdb["price_epic"]  = np.nan
igdb["discount_percent_steam"] = np.nan
igdb["discount_percent_epic"] = np.nan


In [9]:
# Pick a consistent column order for the unified/intermediate frame
cols = [
    "steam_ID", "epic_ID", "igdb_ID",
    "title", "platform",
    "developers", "publishers_steam","publishers_epic",
    "categories_steam", "categories_epic", "categories_igdb",
    "effective_date_steam", "effective_date_epic", "effective_date_igdb",
    "price_steam", "price_epic", "discount_percent_steam","discount_percent_epic"
]

combined = pd.concat(
    [steam[cols], epic[cols], igdb[cols]],
    ignore_index=True
)

agg_dict = {}

for col in combined.columns:
    if col == "platform":
        agg_dict[col] = merge_platforms  # unique, ordered
    elif col == "title":
        agg_dict[col] = "first"  # group key
    else:
        agg_dict[col] = coalesce  # prefer non-null



In [10]:
final_df = (
     combined
    .groupby("title", as_index=False)
    .agg(agg_dict)
)

final_df = final_df.drop(columns=['Unnamed: 0'], errors='ignore')

# Optionally add IDENTITY-like column
final_df.insert(0, "ID", np.arange(1, len(final_df) + 1))

In [11]:
final_df.head()

,ID,steam_ID,epic_ID,igdb_ID,title,platform,developers,publishers_steam,publishers_epic,categories_steam,categories_epic,categories_igdb,effective_date_steam,effective_date_epic,effective_date_igdb,price_steam,price_epic,discount_percent_steam,discount_percent_epic
0,1,2299620,None,None,,[steam],['N/A'],['N/A'],None,"[Single-player, Partial Controller Support, St...",[],[],2023-04-21,NaT,NaT,NaN,NaN,NaN,NaN
1,2,2556940,None,None,! shakabula *,[steam],['Skermunkel'],['Skermunkel'],None,"[Single-player, Full controller support, Steam...",[],[],2023-10-13,NaT,NaT,13.88,NaN,0.0,NaN
2,3,449940,None,None,! that bastard is trying to steal our gold !,[steam],['WTFOMGames'],['WTFOMGames'],None,"[Single-player, Steam Trading Cards, Partial C...",[],[],2016-03-03,NaT,NaT,2.77,NaN,0.0,NaN
3,4,1287250,None,None,! wild russia !,[steam],['Andreev Worlds'],['Andreev Worlds'],None,"[Single-player, Steam Achievements, Partial Co...",[],[],2020-04-28,NaT,NaT,16.23,NaN,0.0,NaN
4,5,866510,None,None,!anyway!,[steam],['EYEFRONT'],['EYEFRONT'],None,"[Single-player, Multi-player, Steam Achievemen...",[],[],2018-06-06,NaT,NaT,1.84,NaN,0.0,NaN


In [12]:
final_df.describe()

,ID,effective_date_steam,effective_date_epic,effective_date_igdb,price_steam,price_epic,discount_percent_steam,discount_percent_epic
count,151209.000000,70486,4069,76654,61122.000000,6668.000000,61137.000000,5712.000000
mean,75605.000000,2021-03-07 02:48:51.033112832,2023-02-27 09:49:35.374785280,2021-12-19 17:59:53.800715008,11.242201,19.158350,4.336343,0.166463
min,1.000000,2015-01-01 00:00:00,2015-07-30 00:00:00,1970-12-31 00:00:00,0.460000,0.000000,0.000000,0.000000
25%,37803.000000,2018-10-25 00:00:00,2021-07-21 00:00:00,2019-10-27 00:00:00,2.410000,4.990000,0.000000,0.000000
50%,75605.000000,2021-06-04 00:00:00,2023-08-30 00:00:00,2022-09-25 00:00:00,4.620000,14.990000,0.000000,0.000000
75%,113407.000000,2023-09-29 00:00:00,2024-10-25 00:00:00,2024-08-07 00:00:00,9.250000,24.990000,0.000000,0.213259
max,151209.000000,2025-10-23 00:00:00,2025-12-01 00:00:00,2027-12-31 00:00:00,8058.650000,199.990000,100.000000,1.000000
std,43650.422764,NaN,NaN,NaN,118.100447,20.090791,16.431000,0.299630


In [ ]:
final_df.to_csv('../data/gold/unified.csv')